# Stage 09 — Route-level environmental exposure POC

## Purpose

Stage 08 identified the strongest proof-of-concept H3 cases where:

- observed activity is relatively important;
- activity is relatively persistent;
- access to high-green-view opportunities is comparatively weaker.

Stage 09 now demonstrates the final technical step:

> **For a small number of strongest H3 cases, what environmental conditions are encountered along the route to the selected high-green-view opportunity?**

For each selected H3 case, this notebook:

1. identifies the nearest high-green-view destination;
2. creates a CAFEIN route from the H3 origin to that destination;
3. reports route-level:
   - PM₂.₅,
   - Green View,
   - road noise;
4. compares the fastest bicycle route with an illustrative PM₂.₅-weighted alternative route.

## Critical interpretation caveat

The CAFEIN Helsinki PM₂.₅ raster is a **single FMI-ENFUSER model-hour sample**.

Therefore:

- this notebook demonstrates the **route-exposure machinery**;
- it does **not** estimate historical PM₂.₅ exposure for the Locomizer dates;
- PM₂.₅ values here should be described as **technical sample-hour exposure context**.

The Green View and road-noise layers also come from the CAFEIN Helsinki sample and should be interpreted according to their own source periods and coverage.


## 1. Imports and paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray

import cafein.sampledata.helsinki as helsinki
from cafein import DetailedItineraries, Exposure, StreetNetwork

PROJECT_ROOT = Path("/scratch/project_2010417")

STAGE08_DIR_CANDIDATES = [
    Path("../outputs/v3/08_environmental_relief_access"),
    PROJECT_ROOT / "paper3" / "outputs" / "v3" / "08_environmental_relief_access",
]

TOP_CASES_NAME = "stage08_top_poc_cases.csv"
RELIEF_GPKG_NAME = "stage08_environmental_relief_access.gpkg"
GREEN_GPKG_NAME = "stage08_high_green_opportunities.gpkg"

STREET_NETWORK_CANDIDATES = [
    Path("helsinki-region-streets.cafein"),
    PROJECT_ROOT / "paper3" / "helsinki-region-streets.cafein",
    PROJECT_ROOT / "paper3" / "cafein" / "helsinki-region-streets.cafein",
    PROJECT_ROOT / "paper3" / "networks" / "helsinki-region-streets.cafein",
]

OUTPUT_DIR = Path(
    "../outputs/v3/09_route_level_environmental_exposure"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

N_CASES = 5
PM25_OPTIMIZE_WEIGHT = 0.1

print("Output:", OUTPUT_DIR.resolve())


Output: /scratch/project_2010417/paper3/airqualityontario22-25/cafein-helsinki-exposure-poc/outputs/v3/09_route_level_environmental_exposure


## 2. Locate Stage 08 outputs

In [2]:
def first_existing(directory_candidates, filename):
    checked = []

    for directory in directory_candidates:
        path = directory / filename
        checked.append(path)

        if path.exists():
            print("Using:", path.resolve())
            return path

    raise FileNotFoundError(
        "Could not find required file. Checked:\n"
        + "\n".join(str(p) for p in checked)
    )

top_cases_path = first_existing(
    STAGE08_DIR_CANDIDATES,
    TOP_CASES_NAME,
)

relief_gpkg_path = first_existing(
    STAGE08_DIR_CANDIDATES,
    RELIEF_GPKG_NAME,
)

green_gpkg_path = first_existing(
    STAGE08_DIR_CANDIDATES,
    GREEN_GPKG_NAME,
)


Using: /scratch/project_2010417/paper3/airqualityontario22-25/cafein-helsinki-exposure-poc/outputs/v3/08_environmental_relief_access/stage08_top_poc_cases.csv
Using: /scratch/project_2010417/paper3/airqualityontario22-25/cafein-helsinki-exposure-poc/outputs/v3/08_environmental_relief_access/stage08_environmental_relief_access.gpkg
Using: /scratch/project_2010417/paper3/airqualityontario22-25/cafein-helsinki-exposure-poc/outputs/v3/08_environmental_relief_access/stage08_high_green_opportunities.gpkg


## 3. Load the strongest Stage 08 cases

In [3]:
top_cases = pd.read_csv(
    top_cases_path
)

h3_relief = gpd.read_file(
    relief_gpkg_path,
    layer="h3_environmental_relief_access",
)

green_destinations = gpd.read_file(
    green_gpkg_path,
    layer="high_green_opportunities",
)

selected_cases = (
    top_cases
    .sort_values(
        "behaviour_green_priority",
        ascending=False,
    )
    .head(N_CASES)
    .copy()
)

display(
    selected_cases[
        [
            "id",
            "persistence_rate",
            "activity_share_A",
            "green_walk_nearest_min",
            "green_bike_nearest_min",
            "green_relief_burden",
            "behaviour_green_priority",
        ]
    ]
)


,id,persistence_rate,activity_share_A,green_walk_nearest_min,green_bike_nearest_min,green_relief_burden,behaviour_green_priority
0,891126d4693ffff,0.618785,0.001106,74.0,19,0.900000,0.579864
1,891126d7577ffff,0.644351,0.000595,62.0,16,0.862500,0.561530
2,8908996d493ffff,0.622549,0.001276,56.0,14,0.816667,0.552695
3,8908996988bffff,0.638743,0.000851,47.0,12,0.750000,0.529287
4,8908996d147ffff,0.688119,0.000425,51.0,14,0.795833,0.503728


## 4. Recover H3 origin geometries

In [4]:
selected_h3 = (
    h3_relief.loc[
        h3_relief["id"].isin(
            selected_cases["id"]
        )
    ]
    .copy()
)

selected_h3 = selected_h3.merge(
    selected_cases[
        [
            "id",
            "behaviour_green_priority",
            "persistence_rate",
            "activity_share_A",
        ]
    ],
    on="id",
    how="left",
)

selected_h3 = gpd.GeoDataFrame(
    selected_h3,
    geometry="geometry",
    crs=h3_relief.crs,
)

origin_points = selected_h3.to_crs(
    "EPSG:4326"
).copy()

origin_points["geometry"] = (
    origin_points.geometry.representative_point()
)

display(
    origin_points[
        [
            "id",
            "behaviour_green_priority",
            "persistence_rate",
            "geometry",
        ]
    ]
)


KeyError: "['behaviour_green_priority', 'persistence_rate'] not in index"

## 5. Match each selected H3 origin to the nearest high-green destination

Stage 08 already calculated travel-time accessibility.

For Stage 09, we need one actual destination per H3 case so we can build a route.

For a compact POC, this notebook selects the **geographically nearest** high-green opportunity point. The route itself is then computed on the CAFEIN street network.


In [ ]:
orig_proj = origin_points.to_crs(
    "EPSG:3067"
)

green_proj = green_destinations.to_crs(
    "EPSG:3067"
)

nearest_pairs = gpd.sjoin_nearest(
    orig_proj[
        [
            "id",
            "geometry",
        ]
    ],
    green_proj[
        [
            "id",
            "Comb_GVI",
            "geometry",
        ]
    ].rename(
        columns={
            "id": "green_id"
        }
    ),
    how="left",
    distance_col="straight_line_distance_m",
)

nearest_pairs = nearest_pairs.drop_duplicates(
    subset="id"
)

display(
    nearest_pairs[
        [
            "id",
            "green_id",
            "Comb_GVI",
            "straight_line_distance_m",
        ]
    ]
)


## 6. Create per-case destination GeoDataFrames

In [ ]:
case_pairs = (
    origin_points[
        [
            "id",
            "geometry",
            "behaviour_green_priority",
            "persistence_rate",
            "activity_share_A",
        ]
    ]
    .merge(
        nearest_pairs[
            [
                "id",
                "green_id",
                "Comb_GVI",
                "straight_line_distance_m",
            ]
        ],
        on="id",
        how="left",
    )
)

case_pairs = case_pairs.merge(
    green_destinations[
        [
            "id",
            "geometry",
        ]
    ].rename(
        columns={
            "id": "green_id",
            "geometry": "destination_geometry",
        }
    ),
    on="green_id",
    how="left",
)

display(case_pairs)


## 7. Load or build the CAFEIN street network

In [ ]:
street_path = None

for candidate in STREET_NETWORK_CANDIDATES:
    if candidate.exists():
        street_path = candidate
        break

if street_path is not None:
    print("Loading:", street_path.resolve())
    streets = StreetNetwork.load(
        street_path,
        mmap=True,
    )

else:
    print(
        "No saved street network found. "
        "Building one from the CAFEIN Helsinki sample."
    )

    streets = StreetNetwork.from_osm(
        helsinki.osm_pbf,
        modes=(
            "walk",
            "bicycle",
            "e_scooter",
            "wheelchair",
            "car",
        ),
        dem=helsinki.dem,
        dem_interval=25,
        country="FI",
    )

print(streets)


## 8. Build the environmental exposure object

The CAFEIN sample includes:

- one model-hour PM₂.₅ raster;
- street-level Green View;
- road-noise polygons.

We attach all three to the same street network.


In [ ]:
green_roads = gpd.read_file(
    helsinki.green_view,
    layer="roads",
)

noise_zones = gpd.read_file(
    helsinki.noise
)

road_noise = noise_zones.loc[
    (noise_zones["source"] == "road")
    &
    (noise_zones["metric"] == "Lden")
].copy()

environment = Exposure(
    streets,
    road_noise=(
        road_noise,
        "db_low",
    ),
    pm25=(
        helsinki.air_quality,
        "PM25Concentration",
    ),
    green_view=(
        green_roads,
        "Comb_GVI",
    ),
    thresholds={
        "road_noise": 55,
    },
    rasterize=10,
)

print("Layers:", environment.layers)
print("Exposure columns:")
print(environment.column_names())


## 9. Route the selected cases

For each H3 case, calculate:

- fastest walking route;
- fastest bicycle route;
- PM₂.₅-weighted bicycle route.

The PM₂.₅ weighting is only an illustrative routing preference.


In [ ]:
route_frames = []

for row in case_pairs.itertuples():
    origin = gpd.GeoDataFrame(
        {
            "id": [
                str(row.id)
            ]
        },
        geometry=[
            row.geometry
        ],
        crs="EPSG:4326",
    )

    destination = gpd.GeoDataFrame(
        {
            "id": [
                str(row.green_id)
            ]
        },
        geometry=[
            row.destination_geometry
        ],
        crs="EPSG:4326",
    )

    fastest_walk = DetailedItineraries(
        streets,
        origin,
        destination,
        transport_mode="walk",
        exposure=environment,
    ).copy()

    fastest_walk["case_id"] = row.id
    fastest_walk["route_choice"] = "fastest_walk"

    fastest_bike = DetailedItineraries(
        streets,
        origin,
        destination,
        transport_mode="bicycle",
        exposure=environment,
    ).copy()

    fastest_bike["case_id"] = row.id
    fastest_bike["route_choice"] = "fastest_bicycle"

    lower_pm25_bike = DetailedItineraries(
        streets,
        origin,
        destination,
        transport_mode="bicycle",
        exposure=environment,
        optimize={
            "pm25": PM25_OPTIMIZE_WEIGHT,
        },
    ).copy()

    lower_pm25_bike["case_id"] = row.id
    lower_pm25_bike["route_choice"] = "pm25_weighted_bicycle"

    route_frames.extend(
        [
            fastest_walk,
            fastest_bike,
            lower_pm25_bike,
        ]
    )

routes = pd.concat(
    route_frames,
    ignore_index=True,
)

routes = gpd.GeoDataFrame(
    routes,
    geometry="geometry",
    crs=route_frames[0].crs,
)

print("Route rows:", len(routes))
display(routes.head())


## 10. Inspect route-level exposure metrics

CAFEIN reports route-level environmental columns such as:

- `pm25_mean`
- `pm25_max`
- `pm25_coverage`
- `green_view_mean`
- `green_view_max`
- `green_view_coverage`
- `road_noise_mean`
- `road_noise_max`
- `road_noise_coverage`
- `road_noise_minutes_above_55`

Coverage should always be checked together with the mean values.


In [ ]:
candidate_columns = [
    "case_id",
    "route_choice",
    "travel_time",
    "distance_m",
    "pm25_mean",
    "pm25_max",
    "pm25_coverage",
    "green_view_mean",
    "green_view_max",
    "green_view_coverage",
    "road_noise_mean",
    "road_noise_max",
    "road_noise_coverage",
    "road_noise_minutes_above_55",
]

route_columns = [
    c
    for c in candidate_columns
    if c in routes.columns
]

display(
    routes[
        route_columns
    ].sort_values(
        [
            "case_id",
            "route_choice",
        ]
    )
)


## 11. Summarize the route alternatives by case

This creates one compact table per case and route choice.


In [ ]:
summary = (
    routes[
        route_columns
    ]
    .copy()
)

summary = summary.merge(
    selected_cases[
        [
            "id",
            "persistence_rate",
            "activity_share_A",
            "green_relief_burden",
            "behaviour_green_priority",
        ]
    ].rename(
        columns={
            "id": "case_id",
        }
    ),
    on="case_id",
    how="left",
)

display(
    summary.sort_values(
        [
            "behaviour_green_priority",
            "case_id",
            "route_choice",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
)


## 12. Compare fastest vs PM₂.₅-weighted bicycle route

For each H3 case, compare:

- additional travel time;
- additional distance;
- change in route-average PM₂.₅.

Remember: this is only a **technical sample-hour demonstration** because the PM₂.₅ raster is not contemporaneous with the Locomizer dates.


In [ ]:
bike_compare = (
    summary.loc[
        summary["route_choice"].isin(
            [
                "fastest_bicycle",
                "pm25_weighted_bicycle",
            ]
        )
    ]
    .pivot(
        index="case_id",
        columns="route_choice",
        values=[
            c
            for c in [
                "travel_time",
                "distance_m",
                "pm25_mean",
                "green_view_mean",
                "road_noise_mean",
            ]
            if c in summary.columns
        ],
    )
)

bike_compare.columns = [
    f"{metric}__{choice}"
    for metric, choice
    in bike_compare.columns
]

bike_compare = bike_compare.reset_index()

if {
    "travel_time__fastest_bicycle",
    "travel_time__pm25_weighted_bicycle",
}.issubset(
    bike_compare.columns
):
    bike_compare["extra_time_min"] = (
        bike_compare[
            "travel_time__pm25_weighted_bicycle"
        ]
        - bike_compare[
            "travel_time__fastest_bicycle"
        ]
    )

if {
    "pm25_mean__fastest_bicycle",
    "pm25_mean__pm25_weighted_bicycle",
}.issubset(
    bike_compare.columns
):
    bike_compare["pm25_mean_reduction"] = (
        bike_compare[
            "pm25_mean__fastest_bicycle"
        ]
        - bike_compare[
            "pm25_mean__pm25_weighted_bicycle"
        ]
    )

display(bike_compare)


## 13. Map the routes

In [ ]:
routes_map = routes.to_crs(
    "EPSG:3067"
)

origins_map = origin_points.to_crs(
    "EPSG:3067"
)

green_map = green_destinations.to_crs(
    "EPSG:3067"
)

for case_id in selected_cases["id"]:
    case_routes = routes_map.loc[
        routes_map["case_id"] == case_id
    ].copy()

    if case_routes.empty:
        continue

    fig, ax = plt.subplots(
        figsize=(9, 8),
        dpi=160,
    )

    green_map.plot(
        ax=ax,
        markersize=5,
        alpha=0.15,
    )

    origins_map.loc[
        origins_map["id"] == case_id
    ].plot(
        ax=ax,
        markersize=55,
        zorder=4,
    )

    case_routes.plot(
        ax=ax,
        column="route_choice",
        categorical=True,
        legend=True,
        linewidth=3,
    )

    minx, miny, maxx, maxy = (
        case_routes.total_bounds
    )

    pad = 800

    ax.set_xlim(
        minx - pad,
        maxx + pad,
    )
    ax.set_ylim(
        miny - pad,
        maxy + pad,
    )

    ax.set_title(
        f"Route exposure POC — {case_id}"
    )
    ax.set_axis_off()

    plt.tight_layout()
    plt.show()


## 14. Read the POC result

For each case, the important questions are:

1. **Behavioural context**
   - Was this H3 cell high-priority because activity was important and persistent?

2. **Environmental-opportunity context**
   - Was access to high-green-view destinations relatively weak?

3. **Route context**
   - How long is the fastest walking/cycling route?
   - What PM₂.₅, Green View, and road-noise conditions are encountered along it?

4. **Alternative-route context**
   - Can a slightly longer bicycle route reduce route-average PM₂.₅ in the sample-hour raster?

The purpose is not to establish historical exposure. The purpose is to prove that observed behavioural screening can be connected all the way through to route-level environmental conditions.


## 15. Final POC chain

At this point the proof-of-concept workflow is complete:

**Stage 06 — Locomizer behavioural response**
→ identify persistent/high-importance H3 cells

**Stage 07 — CAFEIN mobility context**
→ quantify walking/cycling accessibility

**Stage 08 — environmental opportunity access**
→ quantify access to high-green-view opportunities

**Stage 09 — route environmental exposure**
→ quantify PM₂.₅, Green View, and road-noise conditions along selected routes

A substantive future study can replace:
- sample-hour PM₂.₅ with historical event-specific air quality;
- high-green-view streets with validated cleaner-air/cooling/relief facilities;
- the exploratory accessibility burden with a validated transport-poverty framework.


## 16. Save outputs

In [ ]:
routes_csv = (
    OUTPUT_DIR
    / "stage09_route_environmental_exposure.csv"
)

routes_gpkg = (
    OUTPUT_DIR
    / "stage09_route_environmental_exposure.gpkg"
)

summary_csv = (
    OUTPUT_DIR
    / "stage09_route_exposure_summary.csv"
)

bike_compare_csv = (
    OUTPUT_DIR
    / "stage09_bicycle_route_comparison.csv"
)

routes.drop(
    columns="geometry"
).to_csv(
    routes_csv,
    index=False,
)

routes.to_file(
    routes_gpkg,
    layer="route_environmental_exposure",
    driver="GPKG",
)

summary.to_csv(
    summary_csv,
    index=False,
)

bike_compare.to_csv(
    bike_compare_csv,
    index=False,
)

print("Saved:")
for p in [
    routes_csv,
    routes_gpkg,
    summary_csv,
    bike_compare_csv,
]:
    print(" -", p)
